# Phase 2 — Face & Emotion Extractor Raw Features

Tests F-01 through F-06. Run each cell in order.  

## Setup

In [1]:
import sys, hashlib, json
import numpy as np
from pathlib import Path
from unittest.mock import patch, MagicMock
sys.path.insert(0, '/home/tamires/projects/rpp-aevans-ab/tamires/DecomposingMovies')

print(f"Python: {sys.version}")
print(f"NumPy:  {np.__version__}")

# Fixture directory
FIXTURES_DIR = Path("tests/fixtures/vectors")
FIXTURES_DIR.mkdir(parents=True, exist_ok=True)

# Check DeepFace
try:
    from deepface import DeepFace
    print(f"DeepFace OK")
    DEEPFACE_OK = True
except ImportError:
    print("DeepFace not installed — tests will be skipped")
    DEEPFACE_OK = False

# Load our extractor
from cinematic_surprise.modalities.face import FaceExtractor, _extract_frame
from cinematic_surprise.config import EMOTION_LABELS, FACE_MIN_AREA_PCT, DEEPFACE_BACKEND

face_ext = FaceExtractor()
print(f"Backend:       {DEEPFACE_BACKEND}")
print(f"Min area pct:  {FACE_MIN_AREA_PCT}")
print(f"Emotion labels: {EMOTION_LABELS}")
print("\nSetup complete.")

Python: 3.10.12 (main, Mar  3 2026, 11:56:32) [GCC 11.4.0]
NumPy:  1.26.4


I0000 00:00:1774314534.677667 3540494 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774314534.717070 3540494 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774314535.649674 3540494 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


DeepFace OK
Backend:       retinaface
Min area pct:  0.0001
Emotion labels: ['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral']

Setup complete.


## Helper: Generate test frames

In [2]:
import cv2

def make_face_frame(n_faces=1, size=480):
    """
    Generate a frame with simple face-like circles.
    DeepFace will attempt to detect these — real detection depends
    on the backend, so we use real photos when possible.
    """
    img = np.full((size, size, 3), 200, dtype=np.uint8)
    for i in range(n_faces):
        cx = size // (n_faces + 1) * (i + 1)
        cy = size // 2
        r = size // 6
        # Skin-toned circle
        cv2.circle(img, (cx, cy), r, (150, 180, 220), -1)
        # Eyes
        cv2.circle(img, (cx - r//3, cy - r//4), r//8, (50, 50, 50), -1)
        cv2.circle(img, (cx + r//3, cy - r//4), r//8, (50, 50, 50), -1)
        # Mouth
        cv2.ellipse(img, (cx, cy + r//3), (r//4, r//8), 0, 0, 180, (50, 50, 50), 2)
    return img

def make_landscape_frame(size=480):
    """Frame with no faces: horizontal gradient, nature-like."""
    img = np.zeros((size, size, 3), dtype=np.uint8)
    # Sky gradient (top half)
    for y in range(size // 2):
        img[y, :, 0] = int(200 - y * 0.5)  # blue decreasing
        img[y, :, 1] = int(180 - y * 0.3)
        img[y, :, 2] = int(100 + y * 0.2)
    # Green ground (bottom half)
    for y in range(size // 2, size):
        img[y, :, 0] = 50
        img[y, :, 1] = int(100 + (y - size//2) * 0.3)
        img[y, :, 2] = 30
    return img

print("Helper functions ready.")

Helper functions ready.


---
## F-01: Emotion Probabilities Sum to 1 + Face Presence Vector Shape
**Why:** Unnormalized probabilities bias the EMA. Shape check confirms (3,) face presence vector.

In [3]:
if not DEEPFACE_OK:
    print("F-01: SKIPPED — DeepFace not installed")
else:
    # Generate diverse frames and test with real DeepFace
    # We use the full FaceExtractor.extract() which processes a list of frames
    # and returns per-second aggregated results
    
    # Create frames with synthetic face-like content
    test_frames = []
    for seed in range(20):
        rng = np.random.RandomState(seed)
        # Random image that might trigger face detection
        frame = rng.randint(0, 256, (480, 480, 3), dtype=np.uint8)
        test_frames.append(frame)
    
    result = face_ext.extract(test_frames)
    
    # Check emotion vector
    emotion_vec = result["emotion"]  # (7,) per-second average
    faces_vec = result["faces"]       # (3,) [n_faces_mean, n_faces_std, coverage_std]
    
    print(f"Emotion vector shape: {emotion_vec.shape}")
    print(f"Emotion vector: {emotion_vec}")
    print(f"Emotion sum: {emotion_vec.sum():.6f}")
    print(f"All >= 0: {np.all(emotion_vec >= 0)}")
    print()
    print(f"Face presence vector shape: {faces_vec.shape}")
    print(f"Face presence vector: {faces_vec}")
    
    # The emotion vec either sums to ~1 (faces detected) or is all zeros (no faces)
    emo_sum = emotion_vec.sum()
    sum_ok = (abs(emo_sum - 1.0) < 1e-4) or (abs(emo_sum) < 1e-9)
    non_neg = np.all(emotion_vec >= 0)
    shape_emo_ok = emotion_vec.shape == (7,)
    shape_face_ok = faces_vec.shape == (3,)
    
    print()
    print(f"Emotion shape (7,)?    {shape_emo_ok}  {'✓' if shape_emo_ok else '✗'}")
    print(f"Emotion sum valid?     {sum_ok}  {'✓' if sum_ok else '✗'}  (sum={emo_sum:.6f})")
    print(f"All non-negative?      {non_neg}  {'✓' if non_neg else '✗'}")
    print(f"Face shape (3,)?       {shape_face_ok}  {'✓' if shape_face_ok else '✗'}")
    print()
    
    if shape_emo_ok and sum_ok and non_neg and shape_face_ok:
        print("F-01: PASSED ✓")
    else:
        print("F-01: FAILED ✗")

W0000 00:00:1774314540.904197 3540494 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
I0000 00:00:1774314540.905799 3540494 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 7352 MB memory:  -> device: 0, name: NVIDIA H100 80GB HBM3 MIG 1g.10gb, pci bus id: 0000:19:00.0, compute capability: 9.0a
I0000 00:00:1774314544.729372 3542530 cuda_dnn.cc:461] Loaded cuDNN version 91002
W0000 00:00:1774314548.105095 3542530 nvptx_libdevice_path.cc:41] Can't find libdevice directory ${CUDA_DIR}/nvvm/libdevice. This may result in compilation or runtime failures, if the program we try to run uses routines from libdevice.
Searched for CUDA in the following directories:
  ./cuda_sdk_lib
  ipykernel_launcher.runfiles/cuda_nvcc
  ipykernel_launcher.runfiles/cuda_nvdisasm
  ipykernel_launcher.runfiles/nvidia_nvshmem
  ipykernel_launcher.runfiles/cuda_nvvm
  ipykern

Emotion vector shape: (7,)
Emotion vector: [8.11862602e-05 3.29945938e-05 2.27359793e-01 2.49696957e-02
 2.18401372e-01 1.65134211e-03 5.27503618e-01]
Emotion sum: 1.000000
All >= 0: True

Face presence vector shape: (3,)
Face presence vector: [1. 0. 0.]

Emotion shape (7,)?    True  ✓
Emotion sum valid?     True  ✓  (sum=1.000000)
All non-negative?      True  ✓
Face shape (3,)?       True  ✓

F-01: PASSED ✓


---
## F-02: Coverage Weighting Formula
**Why:** Wrong weighting gives equal influence to background and foreground faces.

In [4]:
# Test with mocked DeepFace output to verify the weighting math exactly
# We bypass DeepFace and inject known face detections

# Two faces with known areas and emotions:
# Face 1: large (200x200), mostly happy
# Face 2: small (50x50), mostly angry

frame = np.zeros((480, 480, 3), dtype=np.uint8)
frame_area = 480 * 480

mock_results = [
    {
        "region": {"x": 50, "y": 50, "w": 200, "h": 200},
        "emotion": {
            "angry": 5.0, "disgust": 2.0, "fear": 3.0,
            "happy": 70.0, "sad": 5.0, "surprise": 10.0, "neutral": 5.0
        }
    },
    {
        "region": {"x": 300, "y": 300, "w": 50, "h": 50},
        "emotion": {
            "angry": 80.0, "disgust": 5.0, "fear": 3.0,
            "happy": 2.0, "sad": 5.0, "surprise": 3.0, "neutral": 2.0
        }
    },
]

# Hand-calculate expected result
area1 = 200 * 200  # = 40000
area2 = 50 * 50    # = 2500
w1 = area1 / (area1 + area2)  # = 40000/42500 ≈ 0.9412
w2 = area2 / (area1 + area2)  # = 2500/42500  ≈ 0.0588

# Normalize each emotion vector to sum to 1
emo1_raw = np.array([5, 2, 3, 70, 5, 10, 5], dtype=np.float32)
emo2_raw = np.array([80, 5, 3, 2, 5, 3, 2], dtype=np.float32)
emo1 = emo1_raw / emo1_raw.sum()
emo2 = emo2_raw / emo2_raw.sum()

expected = w1 * emo1 + w2 * emo2

print(f"Area 1: {area1}, Area 2: {area2}")
print(f"Weight 1: {w1:.4f}, Weight 2: {w2:.4f}")
print(f"Emo1 (normalized): {emo1}")
print(f"Emo2 (normalized): {emo2}")
print(f"Expected weighted avg: {expected}")
print(f"Expected sum: {expected.sum():.6f}")
print()

# Run through _extract_frame with mocked DeepFace
with patch('cinematic_surprise.modalities.face.DeepFace') as mock_df:
    mock_df.analyze.return_value = mock_results
    actual_emo, actual_nfaces, actual_cov = _extract_frame(frame)

print(f"Actual weighted avg:   {actual_emo}")
print(f"Actual n_faces:        {actual_nfaces}")
print(f"Actual coverage:       {actual_cov:.4f}%")
print()

# Check
diff = np.abs(actual_emo - expected).max()
nfaces_ok = actual_nfaces == 2.0
emo_ok = diff < 1e-5
expected_cov = (area1 + area2) / frame_area * 100.0
cov_ok = abs(actual_cov - expected_cov) < 0.01

print(f"Emotion max diff:      {diff:.2e}  {'✓' if emo_ok else '✗'}")
print(f"n_faces == 2?          {nfaces_ok}  {'✓' if nfaces_ok else '✗'}")
print(f"Coverage: {actual_cov:.4f} vs {expected_cov:.4f}  {'✓' if cov_ok else '✗'}")
print()

if emo_ok and nfaces_ok and cov_ok:
    print("F-02: PASSED ✓")
else:
    print("F-02: FAILED ✗")

Area 1: 40000, Area 2: 2500
Weight 1: 0.9412, Weight 2: 0.0588
Emo1 (normalized): [0.05 0.02 0.03 0.7  0.05 0.1  0.05]
Emo2 (normalized): [0.8  0.05 0.03 0.02 0.05 0.03 0.02]
Expected weighted avg: [0.09411765 0.0217647  0.03       0.66       0.05       0.09588236
 0.04823529]
Expected sum: 1.000000

Actual weighted avg:   [0.09411765 0.02176471 0.03       0.65999999 0.05       0.09588235
 0.04823529]
Actual n_faces:        2.0
Actual coverage:       18.4462%

Emotion max diff:      3.75e-08  ✓
n_faces == 2?          True  ✓
Coverage: 18.4462 vs 18.4462  ✓

F-02: PASSED ✓


---
## F-03: Zero-Face Returns Exact Zero Vector
**Why:** Non-zero output when no faces present kills the face-disappearance surprise signal.

In [5]:
# Test with mocked DeepFace returning no detections
landscape = make_landscape_frame(480)

with patch('cinematic_surprise.modalities.face.DeepFace') as mock_df:
    # Simulate no face detection
    mock_df.analyze.return_value = []
    emo_vec, n_faces, coverage = _extract_frame(landscape)

print(f"Emotion vector: {emo_vec}")
print(f"n_faces:        {n_faces}")
print(f"Coverage:       {coverage}")
print()

is_zero = np.allclose(emo_vec, 0.0)
nf_zero = n_faces == 0.0
cov_zero = coverage == 0.0
no_nan = not np.any(np.isnan(emo_vec))

print(f"Emotion is zero?  {is_zero}  {'✓' if is_zero else '✗'}")
print(f"n_faces == 0?     {nf_zero}  {'✓' if nf_zero else '✗'}")
print(f"coverage == 0?    {cov_zero}  {'✓' if cov_zero else '✗'}")
print(f"No NaN?           {no_nan}  {'✓' if no_nan else '✗'}")
print()

if is_zero and nf_zero and cov_zero and no_nan:
    print("F-03: PASSED ✓")
else:
    print("F-03: FAILED ✗")

Emotion vector: [0. 0. 0. 0. 0. 0. 0.]
n_faces:        0.0
Coverage:       0.0

Emotion is zero?  True  ✓
n_faces == 0?     True  ✓
coverage == 0?    True  ✓
No NaN?           True  ✓

F-03: PASSED ✓


---
## F-04: False Positive Filtering (Min Area)
**Why:** Without filtering, textured backgrounds register as faces and inject phantom emotion signal.

In [6]:
frame = np.zeros((480, 480, 3), dtype=np.uint8)
frame_area = 480 * 480

# Mock: one face below threshold, one above
tiny_area = frame_area * FACE_MIN_AREA_PCT * 0.5   # half the threshold → should be filtered
tiny_side = int(np.sqrt(tiny_area))
normal_side = 100  # well above threshold

mock_results = [
    {
        "region": {"x": 10, "y": 10, "w": tiny_side, "h": tiny_side},
        "emotion": {
            "angry": 90.0, "disgust": 2.0, "fear": 2.0,
            "happy": 2.0, "sad": 2.0, "surprise": 1.0, "neutral": 1.0
        }
    },
    {
        "region": {"x": 200, "y": 200, "w": normal_side, "h": normal_side},
        "emotion": {
            "angry": 2.0, "disgust": 1.0, "fear": 1.0,
            "happy": 80.0, "sad": 5.0, "surprise": 6.0, "neutral": 5.0
        }
    },
]

print(f"FACE_MIN_AREA_PCT: {FACE_MIN_AREA_PCT}")
print(f"Frame area: {frame_area}")
print(f"Min pixel area: {frame_area * FACE_MIN_AREA_PCT:.1f}")
print(f"Tiny face area: {tiny_side * tiny_side} (should be FILTERED)")
print(f"Normal face area: {normal_side * normal_side} (should be KEPT)")
print()

with patch('cinematic_surprise.modalities.face.DeepFace') as mock_df:
    mock_df.analyze.return_value = mock_results
    emo_vec, n_faces, coverage = _extract_frame(frame)

print(f"n_faces detected: {n_faces}")
print(f"Emotion vector: {emo_vec}")
print()

# Should have only 1 face (the normal one), not 2
nf_ok = n_faces == 1.0
# The emotion should be dominated by happy (the normal face), not angry (the tiny one)
happy_idx = 3  # EMOTION_LABELS = ["angry", "disgust", "fear", "happy", "sad", "surprise", "neutral"]
dominant = np.argmax(emo_vec)
dominant_ok = dominant == happy_idx

print(f"Only 1 face kept?        {nf_ok}  {'✓' if nf_ok else '✗'}")
print(f"Dominant = happy (idx 3)? {dominant_ok}  {'✓' if dominant_ok else '✗'}  (got idx {dominant})")
print()

if nf_ok and dominant_ok:
    print("F-04: PASSED ✓")
else:
    print("F-04: FAILED ✗")

FACE_MIN_AREA_PCT: 0.0001
Frame area: 230400
Min pixel area: 23.0
Tiny face area: 9 (should be FILTERED)
Normal face area: 10000 (should be KEPT)

n_faces detected: 1.0
Emotion vector: [0.02       0.01       0.01       0.80000001 0.05       0.06
 0.05      ]

Only 1 face kept?        True  ✓
Dominant = happy (idx 3)? True  ✓  (got idx 3)

F-04: PASSED ✓


---
## F-05: Backend Determinism
**Why:** Non-deterministic detection means surprise contains detector noise, not stimulus signal.

In [7]:
if not DEEPFACE_OK:
    print("F-05: SKIPPED — DeepFace not installed")
else:
    # Use a real frame and run DeepFace twice
    rng = np.random.RandomState(77)
    test_frame = rng.randint(0, 256, (480, 480, 3), dtype=np.uint8)
    
    emo_a, nf_a, cov_a = _extract_frame(test_frame)
    emo_b, nf_b, cov_b = _extract_frame(test_frame)
    
    emo_match = np.array_equal(emo_a, emo_b)
    nf_match = nf_a == nf_b
    cov_match = cov_a == cov_b
    
    print(f"Run A: n_faces={nf_a}, emo={emo_a[:3]}..., cov={cov_a:.4f}")
    print(f"Run B: n_faces={nf_b}, emo={emo_b[:3]}..., cov={cov_b:.4f}")
    print()
    print(f"Emotion identical?   {emo_match}  {'✓' if emo_match else '✗'}")
    print(f"n_faces identical?   {nf_match}  {'✓' if nf_match else '✗'}")
    print(f"Coverage identical?  {cov_match}  {'✓' if cov_match else '✗'}")
    print()
    
    if emo_match and nf_match and cov_match:
        print("F-05: PASSED ✓")
    else:
        emo_diff = np.abs(emo_a - emo_b).max()
        print(f"F-05: FAILED ✗ — Max emotion diff = {emo_diff:.2e}")

Run A: n_faces=1.0, emo=[2.25523036e-06 3.08488382e-08 6.45355806e-02]..., cov=99.5838
Run B: n_faces=1.0, emo=[2.25523036e-06 3.08488382e-08 6.45355806e-02]..., cov=99.5838

Emotion identical?   True  ✓
n_faces identical?   True  ✓
Coverage identical?  True  ✓

F-05: PASSED ✓


---
## F-06: Pin DeepFace Version + Save Reference Outputs
**Why:** DeepFace silently downloads models. Version drift changes detection behavior.

In [10]:
pin_file = FIXTURES_DIR / "deepface_pin.json"

# Collect version info
current_info = {
    "backend": DEEPFACE_BACKEND,
    "min_area_pct": FACE_MIN_AREA_PCT,
    "emotion_labels": EMOTION_LABELS,
}

if DEEPFACE_OK:
    try:
        import deepface
        current_info["deepface_version"] = deepface.__version__
    except AttributeError:
        current_info["deepface_version"] = "unknown"

print("Current configuration:")
for k, v in current_info.items():
    print(f"  {k}: {v}")

# Save reference outputs for 10 deterministic frames
ref_file = FIXTURES_DIR / "face_reference_outputs.npz"

ref_emotions = []
ref_nfaces = []
ref_coverages = []

for seed in range(10):
    rng = np.random.RandomState(seed * 100)
    frame = rng.randint(0, 256, (480, 480, 3), dtype=np.uint8)
    emo, nf, cov = _extract_frame(frame)
    ref_emotions.append(emo)
    ref_nfaces.append(nf)
    ref_coverages.append(cov)

ref_emotions = np.stack(ref_emotions)    # (10, 7)
ref_nfaces = np.array(ref_nfaces)        # (10,)
ref_coverages = np.array(ref_coverages)  # (10,)

print(f"\nReference outputs computed for 10 frames:")
print(f"  Emotions shape: {ref_emotions.shape}")
print(f"  Faces detected: {ref_nfaces}")
print(f"  Total faces across 10 frames: {ref_nfaces.sum():.0f}")

if pin_file.exists():
    with open(pin_file, "r") as f:
        saved_info = json.load(f)
    ok = True
    for key in current_info:
        if key in saved_info and str(current_info[key]) != str(saved_info[key]):
            print(f"  MISMATCH: {key} saved={saved_info[key]} current={current_info[key]}")
            ok = False
    if ok:
        print("\nConfig matches saved pin.")
    else:
        print("\nConfig DRIFTED from saved pin.")
else:
    with open(pin_file, "w") as f:
        json.dump(current_info, f, indent=2)
    print(f"\nSaved config pin to {pin_file}")

if ref_file.exists():
    saved = dict(np.load(ref_file))
    emo_match = np.allclose(ref_emotions, saved["emotions"], atol=1e-5)
    nf_match = np.array_equal(ref_nfaces, saved["nfaces"])
    print(f"\nEmotions match reference?  {emo_match}  {'✓' if emo_match else '✗'}")
    print(f"Face counts match?         {nf_match}  {'✓' if nf_match else '✗'}")
    if emo_match and nf_match:
        print("\nF-06: PASSED ✓")
    else:
        print("\nF-06: FAILED ✗ — Reference outputs have drifted")
else:
    np.savez(ref_file, emotions=ref_emotions, nfaces=ref_nfaces, coverages=ref_coverages)
    print(f"Saved reference outputs to {ref_file}")
    print("\nF-06: FIRST RUN — Re-run this cell to verify.")

Current configuration:
  backend: retinaface
  min_area_pct: 0.0001
  emotion_labels: ['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral']
  deepface_version: 0.0.99

Reference outputs computed for 10 frames:
  Emotions shape: (10, 7)
  Faces detected: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
  Total faces across 10 frames: 10

Config matches saved pin.

Emotions match reference?  True  ✓
Face counts match?         True  ✓

F-06: PASSED ✓
